In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from nltk.tokenize import word_tokenize

In [22]:
sentences = ["moana1 is great",
    "moana2 not great"
]


In [44]:
# ------------------
# Preprocessing
# ------------------


sentence = " ".join(sentences)

tokens = word_tokenize(sentence)
vocab = sorted(list(set(tokens)))

word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for w, i in word_to_idx.items()}


bigrams = []

for s in sentences:
    # Split sentence into individual words
    words = s.split()  # or word_tokenize(s)
    
    # Pair adjacent words together
    pairs = list(zip(words, words[1:]))
    bigrams.extend(pairs)


input_indices = []
target_indices = []
for w1, w2 in bigrams:
    x_idx = word_to_idx[w1]
    y_idx = word_to_idx[w2]
    
    input_indices.append(x_idx)
    target_indices.append(y_idx)

print("Tokens: ", tokens)
print("Vocab: ", vocab)
print("Bigrams: ", bigrams)


Tokens:  ['moana1', 'is', 'great', 'moana2', 'not', 'great']
Vocab:  ['great', 'is', 'moana1', 'moana2', 'not']
Bigrams:  [('moana1', 'is'), ('is', 'great'), ('moana2', 'not'), ('not', 'great')]


In [45]:
# ------------------
# Model Layers
# ------------------

embedding = nn.Embedding(len(vocab), 8)
hidden = nn.Linear(8, 16)
output = nn.Linear(16, len(vocab))

params = (
    list(embedding.parameters())
    + list(hidden.parameters())
    + list(output.parameters())
)

optimizer = optim.Adam(params, lr=0.01)
loss_fn = nn.CrossEntropyLoss()


In [46]:
# ------------------
# Training
# ------------------

X = torch.tensor(input_indices)
y = torch.tensor(target_indices)

for epoch in range(500):

    x = embedding(X)
    x = torch.relu(hidden(x))
    logits = output(x)

    loss = loss_fn(logits, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [47]:
# ------------------
# Prediction
# ------------------

def predict(word):
    idx = torch.tensor([word_to_idx[word]])

    x = embedding(idx)
    x = torch.relu(hidden(x))
    logits = output(x)

    pred = torch.argmax(logits, dim=1).item()

    return idx_to_word[pred]

print("moana1 ->", predict("moana1"))
print("is     ->", predict("is"))
print("moana2 ->", predict("moana2"))
print("not    ->", predict("not"))

moana1 -> is
is     -> great
moana2 -> not
not    -> great
